<a href="https://colab.research.google.com/github/kokami236/osiro1/blob/main/%E6%8E%A8%E8%AB%96%E3%83%87%E3%83%BC%E3%82%BF%E6%95%B4%E5%BD%A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install open3d


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.8 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import open3d as o3d
import numpy as np
import os

# --- ユーザーが設定する項目 ---

# 1. 入力する点群ファイルのパス
input_file = "/content/drive/MyDrive/ジオラマ/唐津城①.ply"

# 2. 中心の点から残したい半径 (単位は点群の座標系に依存します)
# この値を大きくすると、より多くの点が残ります。
radius = 2.2

# 3. 保存するファイル名
output_file = "/content/drive/MyDrive/ジオラマ/唐津城②centerd.ply"

# --------------------------


# ファイルの存在を確認
if not os.path.exists(input_file):
    print(f"エラー: 入力ファイルが見つかりません: {input_file}")
else:
    # 点群データを読み込む
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点群の読み込みに失敗したか、点が含まれていません。")
    else:
        print(f"処理前の点の数: {len(pcd.points)}")

        # 1. 点群の重心（中心）を計算
        center = pcd.get_center()
        print(f"計算された中心座標: {center}")

        # 2. 各点が中心からどれだけ離れているか計算
        points = np.asarray(pcd.points)

        # NumPyを使って全点の中心からの距離を高速に計算
        distances = np.linalg.norm(points - center, axis=1)

        # 3. 指定した半径の内側にある点のインデックスを取得
        indices = np.where(distances <= radius)[0]

        # 4. 半径内の点群だけを抽出して新しい点群オブジェクトを作成
        centered_pcd = pcd.select_by_index(indices)

        print(f"半径 {radius} m 内の点の数: {len(centered_pcd.points)}")

        # 5. 結果をファイルに書き出す
        o3d.io.write_point_cloud(output_file, centered_pcd)
        print(f"処理後のファイルを {output_file} に保存しました。")

処理前の点の数: 1242982
計算された中心座標: [ 0.47941636 -0.16381352  1.20417714]
半径 2.2 m 内の点の数: 927558
処理後のファイルを /content/drive/MyDrive/ジオラマ/唐津城②centerd.ply に保存しました。


学習用データと正規化


In [9]:
# ==============================================================================
# 1. 準備：ライブラリのインストールとドライブのマウント
# ==============================================================================
import os
import sys
import numpy as np
import copy

# Open3Dのインストール（入っていない場合）
try:
    import open3d as o3d
except ImportError:
    os.system("pip install open3d")
    import open3d as o3d

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# ==============================================================================
# 2. 設定：ファイルパスを指定してください
# ==============================================================================

# 【A】基準となるデータ（学習に使用したデータセットの一部、例: kesson2.ply）
# ※ このデータの「大きさ」が正解基準になります。
REFERENCE_PLY_PATH = "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"

# 【B】推論したい新しいデータ（例: 新しいお城、模型など）
# ※ これの大きさを【A】に合わせます。
TARGET_PLY_PATH    = "/content/drive/MyDrive/ジオラマ/唐津城②centerd.ply"

# 【C】保存先ファイル名（これが推論コードのINPUTになります）
OUTPUT_PLY_PATH    = "/content/drive/MyDrive/ジオラマ/ready_for_inference②.ply"


# ==============================================================================
# 3. メイン処理：スケーリングと正規化
# ==============================================================================
def get_diag_size(pcd):
    """点群の対角線の長さ（AABBの大きさ）を計算する関数"""
    pts = np.asarray(pcd.points)
    if len(pts) == 0:
        return 0.0, np.zeros(3)

    min_bound = pts.min(axis=0)
    max_bound = pts.max(axis=0)
    size_vector = max_bound - min_bound
    diag_length = np.linalg.norm(size_vector)

    return diag_length, size_vector

def main():
    print("🚀 処理を開始します...")

    # 1. ファイル読み込み
    if not os.path.exists(REFERENCE_PLY_PATH):
        print(f"❌ 基準ファイルが見つかりません: {REFERENCE_PLY_PATH}")
        return
    if not os.path.exists(TARGET_PLY_PATH):
        print(f"❌ ターゲットファイルが見つかりません: {TARGET_PLY_PATH}")
        return

    print(f"📖 基準データを読み込み中: {os.path.basename(REFERENCE_PLY_PATH)}")
    ref_pcd = o3d.io.read_point_cloud(REFERENCE_PLY_PATH)

    print(f"📖 推論データを読み込み中: {os.path.basename(TARGET_PLY_PATH)}")
    tgt_pcd = o3d.io.read_point_cloud(TARGET_PLY_PATH)

    # 2. サイズ計測
    ref_diag, ref_vec = get_diag_size(ref_pcd)
    tgt_diag, tgt_vec = get_diag_size(tgt_pcd)

    print("-" * 50)
    print(f"📏 [基準] サイズ(対角線): {ref_diag:.4f} m (想定)")
    print(f"📏 [対象] サイズ(対角線): {tgt_diag:.4f} m (想定)")

    if tgt_diag == 0:
        print("❌ エラー: 推論データが空、またはサイズが0です。")
        return

    # 3. スケール倍率の計算 (基準 / 対象)
    scale_factor = ref_diag / tgt_diag
    print(f"⚖️  スケール倍率: {scale_factor:.4f} 倍")
    print("-" * 50)

    # 4. スケール適用
    # ターゲットを複製して編集
    scaled_pcd = copy.deepcopy(tgt_pcd)

    # 重心を中心に拡大縮小
    scaled_pcd.scale(scale_factor, center=scaled_pcd.get_center())

    # 5. 位置合わせ（重要）
    # 推論時のパッチ分割をスムーズにするため、重心を原点(0,0,0)に移動させます
    scaled_pcd.translate(-scaled_pcd.get_center())

    # 念のため基準データも比較用に原点へ移動したと仮定した場合の情報を表示
    print("✅ スケール変換完了")
    print("✅ 重心を原点(0,0,0)に移動しました")

    # 6. 保存
    o3d.io.write_point_cloud(OUTPUT_PLY_PATH, scaled_pcd)
    print(f"\n💾 保存しました: {OUTPUT_PLY_PATH}")
    print("👉 次のステップ: 推論コードの 'INPUT_PLY' にこのパスを指定してください。")

if __name__ == "__main__":
    main()

🚀 処理を開始します...
📖 基準データを読み込み中: 熊本正解データ.ply
📖 推論データを読み込み中: 唐津城②centerd.ply
--------------------------------------------------
📏 [基準] サイズ(対角線): 6.9057 m (想定)
📏 [対象] サイズ(対角線): 6.9916 m (想定)
⚖️  スケール倍率: 0.9877 倍
--------------------------------------------------
✅ スケール変換完了
✅ 重心を原点(0,0,0)に移動しました

💾 保存しました: /content/drive/MyDrive/ジオラマ/ready_for_inference②.ply
👉 次のステップ: 推論コードの 'INPUT_PLY' にこのパスを指定してください。
